In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q -U import_ipynb

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from ixbrl_financials_extract import ixbrl_financials_extract

In [ ]:
import import_ipynb
from OCR import pdf_to_images, run_ocr

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
==((====))==  Unsloth 2026.8.18: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

In [ ]:
# !pip install -q -U unsloth
# !pip install transformers==5.5.0
# !pip install -q -U accelerate bitsandbytes  pymupdf qwen-vl-utils[decord] regex pef
# !pip install -q -U import_ipynb

In [ ]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv, dotenv_values
from requests.exceptions import HTTPError, ConnectionError, Timeout, RequestException
from pathlib import Path

import io
import sys
import time
from collections import Counter

In [ ]:
load_dotenv(Path.cwd() / ".env")

True

In [ ]:
Path.cwd()

PosixPath('/content')

In [ ]:
os.makedirs(Path.cwd() / "company_info_json", exist_ok=True)
os.makedirs(Path.cwd() / "company_filing", exist_ok=True)
os.makedirs(Path.cwd() / "company_ocr_results", exist_ok=True)

In [ ]:
URL_BASE = "https://api.company-information.service.gov.uk"
URL_META_DATA = "https://document-api.company-information.service.gov.uk/document"
TIMEOUT = 10  # seconds

In [ ]:
def get(endpoint, params=None, headers=None):
    response = requests.get(
        endpoint,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    print(f"{endpoint} - {response.status_code}")
    return response.json()

def get_doc_content(endpoint, params=None, headers=None):
    url = f"{endpoint}/content"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    print(f"{endpoint} - {response.status_code} - {response.headers.get('Content-Type')}")
    return response


In [ ]:
def has_xbrl_ixbrl(AA_filing: dict):
  link = AA_filing.get("links", {}).get("document_metadata")

  response = get(link)

  formats = response.get("resources", {})
  if "application/xhtml+xml" in formats or "application/xml" in formats:
      return True

  return False

In [ ]:
IXBRL_MIME = "application/xhtml+xml"
XBRL_MIME  = "application/xml"


def save_json(company_num: str):

  endpoints = {
      "filing":    get(f"{URL_BASE}/company/{company_num}/filing-history"),
      "profile":   get(f"{URL_BASE}/company/{company_num}")
  }

  for name, data in endpoints.items():
      path = Path.cwd() / "company_info_json" / f"{company_num}_{name}.json"
      with open(path, "w") as f:
          json.dump(data, f)


def save_filing_ixbrl(company_num: str, endpoint: str) -> Path:
  """Download the iXBRL (inline XBRL) rendition of a filing.

  The document API serves several renditions of the same document and picks
  between them off the Accept header - with no header you get the PDF back.
  So the header is what actually makes this the iXBRL version rather than a
  PDF written to a file named .xbrl. application/xml is listed as a lower-q
  fallback for older filings that only carry plain XBRL.

  Returns the path written, since the extension depends on what came back.
  """
  response = get_doc_content(
      endpoint,
      headers={"Accept": f"{IXBRL_MIME}, {XBRL_MIME};q=0.9"},
  )

  content_type = response.headers.get("Content-Type", "")
  suffix = ".xhtml" if IXBRL_MIME in content_type else ".xbrl"

  path = Path.cwd() / "company_filing" / f"{company_num}{suffix}"
  with open(path, "wb") as f:
    f.write(response.content)

  return path


def save_filing_pdf(company_num: str, endpoint: str) -> Path:

  response = get_doc_content(endpoint, headers={"Accept": "application/pdf"})

  path = Path.cwd() / "company_filing" / f"{company_num}.pdf"
  with open(path, "wb") as f:
    f.write(response.content)

  return path


In [ ]:
with open(Path.cwd() /  "com_names_n_codes[40000-140000].csv", "r") as f:
  df = pd.read_csv(f)
  company_numbers = df["company_id"].unique()
  company_numbers = company_numbers.tolist()

In [ ]:
company_numbers

['13172868',
 '12176464',
 '16902465',
 '07852962',
 '14568066',
 '15623849',
 '16341698',
 '17119651',
 '16023853',
 '17201396',
 '16356281',
 '16953612',
 '15488786',
 '16499773',
 '13477028',
 '12047798',
 '10379693',
 '13448185',
 '07127657',
 '14361496',
 '10812571',
 '08369001',
 '12367118',
 '16234689',
 'SC454219',
 '16060742',
 '09414744',
 '08162396',
 '13879337',
 '16129186',
 '16219637',
 '16961062',
 '12481302',
 '14025511',
 '16778859',
 '11514585',
 '17059993',
 '10701298',
 '13473756',
 '16019444',
 '15229106',
 '17229431',
 '11746146',
 '17000593',
 '17134211',
 '07152193',
 '12427564',
 '16258260',
 '04182667',
 '07834492',
 '12113137',
 '13726069',
 '13860665',
 '04091297',
 '10921392',
 '02291274',
 '06913353',
 '07487566',
 '05905888',
 '02624500',
 '03337413',
 '14169512',
 '16338068',
 '16538321',
 '16029508',
 '02349961',
 '13720061',
 '14071679',
 '16689326',
 '09543275',
 '11707273',
 '03223962',
 '04465268',
 '01568942',
 '05741183',
 '14765960',
 '02046371',

In [ ]:
Fjson_results = []
for num in company_numbers[60:]:
  save_json(num)

  with open(Path.cwd() / "company_info_json" / f"{num}_profile.json", 'r') as f:
    profile = json.load(f)

  # company needs to be both active and have in date accounts
  if profile["company_status"] != 'active' or profile["accounts"]["overdue"] or profile['company_name'].startswith("!"):
    print(f"{profile['company_name']}({num}) - Not Active or Overdue")

  with open(Path.cwd() / "company_info_json" / f"{num}_filing.json", 'r') as f:
    filing_history = json.load(f)

  # grab the first AA category document
  filing_found = False
  filing_path = None
  is_ixbrl = False
  for i in filing_history["items"]:
    if i["type"] == 'AA':
      is_ixbrl = has_xbrl_ixbrl(i)
      if is_ixbrl:
        filing_path = save_filing_ixbrl(num, i["links"]["document_metadata"])
      else:
        filing_path = save_filing_pdf(num, i["links"]["document_metadata"])
      filing_found = True
      break

  if not filing_found:
    print(f"{profile['company_name']}({num}) - No AA Filing Found")
    continue

  if is_ixbrl:
    financial_tables = ixbrl_financials_extract(filing_path)

    print(financial_tables)
    with open(Path.cwd() / "drive" / "MyDrive" / "company_features" / f"{num}.json", "w") as f:
      json.dump(financial_tables, f)

  else:

    image_paths = pdf_to_images(str(filing_path), Path.cwd() / "drive" / "MyDrive" / "company_filing_pages" / num, dpi = 200)
    results = run_ocr(image_paths)

    Pjson_results = {}

    for key,val in results.items():
      try:
        parsed = json.loads(val)
        Pjson_results[key] = parsed
      except:
        Fjson_results.append(key)

    financial_keywords = [
      "capital", "revenue", "assets", "liabilities",
      "tier 1", "cet1", "income", "expense", "balance",
      "profit", "£", "earning", "financial", "debtor"
    ]

    financial_tables = {}

    for i, item in Pjson_results.items():
      if item["table_found"] == False:
        continue

      for j in item["tables"]:
        try:
          print(f"{i} - {j['table_name']}")

          if not any([key.lower() in j["table_name"].lower() for key in financial_keywords]):
            continue

          print(f"{j['table_name']}")
        except:
          print(f"No Table Name Found for - {j}")

        # check if page is in dict, if so, append to the list, if not, create a new index
        if i in financial_tables.keys():
          financial_tables[i].append(j)
        else:
          financial_tables[i] = [j]

      if i in financial_tables.keys():
        financial_tables[i].append(item)
      else:
        financial_tables[i] = [item]

    financial_tables.values()

    with open(Path.cwd() / "drive" / "MyDrive" / "company_ocr_results" / f"{num}.json", "w") as f:
      json.dump(financial_tables, f)





https://api.company-information.service.gov.uk/company/03337413/filing-history - 200
https://api.company-information.service.gov.uk/company/03337413 - 200
https://document-api.company-information.service.gov.uk/document/bNa1kTZ67y-BujPjblHIsuLu6ttk4luou-xVCTQ7oy0 - 200
https://document-api.company-information.service.gov.uk/document/bNa1kTZ67y-BujPjblHIsuLu6ttk4luou-xVCTQ7oy0 - 200 - application/xhtml+xml
{'2025-03-31': {'CurrentAssets': 2467.0, 'Creditors': 2463.0, 'NetCurrentAssetsLiabilities': 4.0, 'TotalAssetsLessCurrentLiabilities': 4.0, 'NetAssetsLiabilities': 4.0, 'Equity': 4.0, 'AverageNumberEmployeesDuringPeriod': 4.0}, '2024-03-31': {'CurrentAssets': 4867.0, 'Creditors': 4863.0, 'NetCurrentAssetsLiabilities': 4.0, 'TotalAssetsLessCurrentLiabilities': 4.0, 'NetAssetsLiabilities': 4.0, 'Equity': 4.0, 'AverageNumberEmployeesDuringPeriod': 4.0}}
https://api.company-information.service.gov.uk/company/14169512/filing-history - 200
https://api.company-information.service.gov.uk/comp

KeyboardInterrupt: 